# Frozen CUDA evaluation
Requires GPU and Internet. One evaluation attempt; no training. Package versions, model files and inputs are checked before generation. Download evaluation/result.json and evaluation/predictions.json after completion.

In [ ]:
import base64
import hashlib
import io
import os
import pathlib
import subprocess
import sys
import zipfile

raw = base64.b64decode(
    "UEsDBBQAAAAIAFuIVF2HAVEGKQsAAL6SAAALAAAAY2hlY2suanNvbmztnW1v3LgRx9/3UzDGAW6BzcJPbZJ7c3B9bS20ToLERVC0hcGVuCt2JVEVqeztBfnunSGpZ62961q2BPCV7ZVEScOfZ4Z/Drnfjnhw9CM5ytiSZSzx2euT0zdHM3IkRZ75DA/5NAvuaJbxrzTCIzGTkq6YhGP//HaUiUifJbdSsRiP+yJRLFH44VVEpeTLLVEhIwuarHmyIv/NWbadk08sjbZkw1VI2C/UV/CHSBiJ6IJFM5II+DSNaEIVF8mcXEaR2LDAHJY/kvpDzcxfKd3GcN+7JWN3fkizFQvwiAzv8CZBRjc0sge6nydC3WXMF6uES7wuYH7EExbctU6ckZQlAbxG98AmE/AxjUUODyGW5gRok/Gv2KI5zH6BJ0hW7C6jit0tRdZu5+j7jFRWzSXLWjb9ElJ1LMlKgLmg1RC6jXBJ4i1J2EZb4qdmG9gFUlG4utlQo1+///v7b771sPDWsTBiFjbhloT0KwAARlLAhBqKg3eOgxFzgD4BnQCaFLpY5ZKIJaKARgBnEbAIbphtn5CIsxNHxIiJuBYbEgjiEZVRf60ti69PtiInEs4hSpCYPSUPp46HEfNwY3wB2VBJZJ6mAp4XGdDGgDdb5AoiiUyO1VMyceaYGDET6CMiaIrIUORRQDh4BbpmBNqqpxEVJoQnaP+YxJRHPxEddey1HnobvoSfCYNIQ+zTQptPydO542nEPHkmF10wloCf4QpNKhAGSjaMrefEk1UgkopHEfFFDGc9JSIXDpERI1K6nEAw2XA4tOFvZMjTJ6TiD32SRo/JHSEvTsgVTXSSqs0D8QaHuJ5OW+xLwxHIaVH4SOBArvMY61QOIaav93fR0yeCOHrGSE+TFErgdQ0pi3yL5pUqXy6NXe3oeD4QM32CiWNmCsyEPAiAmBIdj6SUB8awAxLzpk9QccSMkZhbLbojNRCsTDgyL9WIStbBDERLn9ziaBkjLV8ggzHaLEBjQhImvLlE21pIBspc3vQJMI6SMVLiWSUlOVa1YXQRlVCegzOYBQjn/Q4S8w9hpk9kccyMkRn0LBiFvFbG23EvA5HSp7U4UsZKCo0BlBVTWpczoIgEftGai7XDMD7ltFtS0mdgx8mLc+KR/+QSAxC6kkiINaG6pAAnlRkaYE6+FDUHXhmiEnQ9ldPB/ri8vdmbpV4WdpHU0WIcSeMkqcp4M0alSFoqXjlyagasQdnpaDKOnVGyA+NrCEjgh5ZsQxSH3iDecZESK1FAMmsjRap7IFZzco1+agsdTlIRcZ8zWbqrjK3AothZcKYchLduiYzjbZS86YlsJAlt2tBz0DEVDel8Guy5/8DrIFg6go6DZaSwbEnAA5NLFwOuTMSDhq5uJY2jYwJ0NIblGK5UKPJVqHDmG/3NMmP71+AdxEtHxHG8jJKXmjBsQw5kytX1AyUmHeHG0TFKOgpRWFf3d1XhavCEDdew2XuK8hBqukU095nV0fPi9OhpymM9SZknNIfYk/FfwTINWB5HSquvdxHzgFjjiBkZMZ9FzNBQIU5tE1+k29o6EvLqFZb5siVPuGLR1pZwmvVH4J4wiuE8lbSNaCUxpgHGtt+fnPwrPzmh521PNSPXf/rbx1eDUviA7OMoHBmFl4gLSeEuQRsXnTVXYOlxF+BJfR+faWbnSSEBx2nUGGtLuZrDJzQmacaUAmhzwLQgGjk3EVXEKbSFliGv4ZipQQxZlB6ySuYxdHbLfhydo6bTI6FItaqoObGMzEndd0oF1yeIGbQYoUyQYA63IbZEHt6W+7pwlSrsm20tuVN0XSpNbcAfmdftSeIDCpQjcXQkXt6QG+/zZ+/9X8jV5edr8udPH+CTf5A/Xr7/K7m8uvrw9/e3+uenn/Gc2w948PLjx0ExekCqchiNDKPLTpAtw2KABtROqgyqgRjWCT0gXDl6RkYPTsElqsyoIGD9CuBkHFVOaAAse6MjmZ5egYQsIQJyNJ7gqFQEYEHha5uWoVEvHwwZNiaWw7L2gAzmWBsZa61K2rbbaiT+QybtZ+16pl0mc8y8ODMeZukonxbX1DNrO2W3Hyo7O3kXJG8dJBOB5JrPjEiQJ3QRMawxgbPKoAbRCiiZ6xAFg7aMV6s79JL4jcDjUqsPCwHG138Vr6sliCst4+OgMY0Yhbb9kPnr2tBuT391MITvHIQTgVBv0dFVu4oHnhMsvRwIk3apksNktJh4xxC3Mr2aFUvhUOPs1Yq0N+JK6rnDyJZ9o1B1zaJ0T8H9YIxOHUYTwejWpD5kUyw6omgaAASpauZKA7Fy5liZCCuezXmADF3VhFHKeJdGhBrKqZw7UCYCSlH9ZsdVOPTyWYRBSWU0kWBJMNhAlFw4SiZCSbHFix0JKRZFGHfMTi8+qonoZbrD9IFy33Zd0w7zOHBeHByvKCcJWCokV3Y/Ou1qcIaV6z0MdcLrA1sZ23Ph664e3wXMWwfMVIDRcwpmfyhIWmiUMRpsZ/VhkqlikqHYSJKnBPixTw0DbtyuDCf566MqZq+wZ+3plA5F7J1DbBqIwWg6EigN4mpZrMlVdVrQK+GuDXoOldazIFJLqnHrzBK7WhGT8WQihkCpbBu1FnApLiuWVZo6ErMqDn+JqR+CwSoRstjuihTRV1dH6WqodsNmcz6zoTgczHDJAnDvU9REGxWAxTMWu33qO2Pc3nPe7sD/inatlPuvGO9/xXZWJ7wUFLA41PhiiM8Rx81BdeA+lkRSLWrp3wvnSr6gq8WV5mXRO5a7q5DvuzLiUMROHWLTQKxPlKCm+lgnhDY39LkCzwfXgKMrkEMHe7+T1MuNJZbA8HsyBLxHrTiPfLRTO2bUi/lpHLOAU+0r9T4tNd9ZFAIWztNWA2r3uaeScijbZ47tabAdFtvkbpC8uDsxBP6wYBDaH8gVnjtcpoFLVRdjQKnncpXz0tv+yEZOue++c4eSc+HImQY5fSlaEUlp0htHbfyMa18oUU1R46xSnT4uWwA24+frshqiCJmDRL7zdr3WvdZ1WI4AS8RKJGCvFbSckB/OTghdInvV1HcJ7A+nJyf7YXN/v++C562DZ0rwYDGXzcTt4hykJWNgVllMXlJ/DSaThNe/ssLeIqlPPWhBRc916m9X089IzCvMdV1YmkbcVjBjsUXh57A/zWmNexSDWKuq7CmTPA7bdw7bKWGrY60ki0ysGRjGU6W6hi5QUwzGXuYQTC1Yvz23qyTl/HeEYF0QwqMDq9mgGvtAO9HT4sRKbcFZN1MRhAPVFTW3qFrU3+SDeqZl0sfB6mbP8PwoYNs1aA7YcQNbSdRUrq0sVzo9sbRDEbN4qLj7kA6vXXzm+Bk7Pz1Jnt1AvyHtDZzftQvRHDdj56Y7Zr0oAxzmapopgMmnENJEDpnaz9x+9aViVAtqh2zk9ziqzh1VU6LqthDPPAxndp6p2OEEvxfTJPBVtw9HzoUjZ0rkePUZHNYYIpZpUL2MrZqCx+HpkD6oXdm27xs7pl6aqVu9Ea1phWArRmflS923qkiTUJLApg9haI+H20FTrw7maBo/TcW6RZHj4kZtP/BAPPFFBjdTBrAF0xNEqGeB0TRgjS3ZzXVwe8ZXCfHzDLHYt9Dn/0WvV8ty6E0BPRXyZF04LHMlKxZh609xPwiMk/gk2dbuwd12ffp5nge2dpWZg21qsHURKtQpm+GXGG5Mam8d4TPx1atTOb7Gz1c5RqSLTNCgxRiXVfR8Rm/Vq145mqZBUwFMhyQzimy5LUz3DXrPxFavhuXYGj9bXrnFZLztyaaq1L81kHwmrnoVLsfVFLgqU6eqkh/nlLspl2QslijSL9hg8fB/UEsDBBQAAAAIAFuIVF0SzYRzoAMAAJwJAAARAAAAY29yZS9ldmFsL2RhdGEucHmdVU2P2zYQvetXTJVDJEArIDkFSlw0QPYStGmBFL0YhkFLI5uNRAoklV1ns/+9w09btgsE0UXQcGb43psP5Xl+/5UNMzNcCsBHNk4DauiVHIFBe2AGPn7+89Pv0PMBG5jIPhkYUWu2R12Bwh4VihYr0HJWLdZ5nmeZi99u+9nMCrdb4OMklQEmhDTuKp1lwfavlsL7d8ywdmBaE4BwmEzeY2LmMPBdPP2LPv2BOU5c7KP9vThm2R8BI6xg4NqsO96atTaKgBq12WRZ9lvKXlCSbyhWf6sZy8yZ4N5r0WRAz8AFsefCALxwHyDmcYeKTGAO6NSpoJfKfSkMbDtnYhZ3e3CJvIANJHSUD7+iOkZNYTcbl4RAGBeCjxO2BrvGArf+/kLBBiCcRI2JVJEq3B+KQhD0AyqXxpfHJ/kOnyRxWPkXpXw4UICL5WIiAC0b0TVBZY9EyNpK1YFmRype1mEPW6634eIivBsrfgl3v8JOysGLp5C6QEDhPuzDNRcWdpvCKrDlKZOHle7aq96jKXIlB8xLV8YfC2ilMCjMIqaMFLwshSfXwKlLiMfGETkJ1pwJSeL5GH+HN+Y+Oe+jE9de4gt455EjGmYb0cJzIjSJ1MVN65Pv5satQeZ4c39+oTd6/oCDRgcraKCQdds4+oWdsdgldsCcBm6CwkRsPMDcbg7buKEvZH+9MKh5pPatnPqbJLnVo3Xmsr6P6UZ2BGG7kBvtm/KlTtwEROpA85VkqYPxrb3Bz45S5LAjhl86+SBgr+Q8adgdKW0dafjBJKqktGXsJCjD7IWFuIL1xlkeOPlZh1pOKApiIDtaPat8Nv3dm7ykmYT+VEI7/3ZfbIWs/OIg8EjLAxUzWPS2JEyZ1auzsocWolXpImqqBZ+KCwf72MbmYsbFgVHHa88g6spt23qQrNOFzV0uPPGxxclv5NoW8QNFdXjvNCRaeCMv41Tff+gH4t2KPn+y4jw3T4H1c+OI2HRQPGE96v1zSTK5tb1EPp429nJAvJmClk3tncLUnHX1DR2vV4OuXFOXtoEGKmQ0l/AOXv8sUcRO23Xv/5J26JkI/Z2Xt3CxYShubNHS9U2aGJGkudEEP4SMpWQeo92hDl/ajZfo4o3ru1ebtd+5G/hlBXn65+Q/iSX+3M43gtMiJb7sSj+DNZto4rri6tawl64P7JOGLxFqiFF1wS/KQAfLP0J5lXRpSQvfMohIT8L8vyAUkPyXCzxas/8AUEsDBBQAAAAIAFuIVF3KEpH+QgkAALYZAAAVAAAAY29yZS9ldmFsL2dlbmVyYXRlLnB5tVhtb9w2Ev6uX8HTfYjUk5Vr+uWwBxXwpfahuDg1UudaYLEQuBLXy6xW1ImU7W2a/96ZISlR8ubaAq2BNityOJyXZ94Yx/G/eyHqE+OtfhS9ZmrHONtyLdhR1aJhqofvN+rdJeM174zoM6ZawXpRqb5mneiZeOLHrhF5FL0bWmDQwoEbXv1LqQNLbm6/TzP2+v03l8jp9e37nBnVV/uMmR6u3Kn+iLfyFnntDOO9YPLYqd6IOlJtc2LbEzN7wRrFa6DMmFb0fS9a0XMj4bpGqY71eLdsmRHaaPYozR7E0AYYX8Aq6QIS3vbq2Bl7Xw0q1LK9Z0duqj37oLb6pXjgTdmoijcvQa3KlLSXd6fxXg16CzITfEXdsG2k3ouabUVb7Y+8P7CK9/2JqQfRr+hEtecGxAIbcSPcQVgU1aFTsjXscS9aJg3bc5ArQurSU+cfZPuBg632sj2gqGq3y5bqd6QTuKcWdcburTu9chFqCvcflTbsq1esFY/ggIMAYw1tI7Qmbqj2wMExTAuwzpE/lUBYWkIw2x3+YNdgEAWqaf4AFngU8n5vWMuPwtpbDYbdXl3fvdAsRgCV1uj2/zGIKXbyiSyPl0SylQZM3fPScgJJFNvxRguy9S2g4YaO7kC/Ek4DYGQLln5UQ1OzRoAUDDBUq6OTBXkysFslcoJL6RBLH1ZRfxWACgXPSBzOjlJrNC/h3CkmESWR6HvV51Ecx1GEgrCy3A1m6EVZOpwCVasMuUJHkVv7oFVr6SvVNKKi3ZxvK3/oW5CLbxthiTpu9o3c+s1b+LQb5tShXG79sj1FnmsvcvRaXnPD/f6VjcQlyRFMJyvtqW7oc0lkA1ovWL0TemhMFH1zdX35/s1deXP5Y/n26ofy7rv/XL39nhUAqej23dX1tz/C7zNej6KoFjvWyepQ1uJBViJBu68gMns8wQej4pRdfI0Lq4jBn9yRa9hf/LZdxj/AwNC3tG1JrayUUCJ/mL7yaqh5LnXJH7hs0NBJ+oxPjERxFK4cOx1PTLa8AtzXOoflBTMmAKjAoRu8jjsIvLLiENiJ1ZSUJN3eQsYclbObrCjsbSvG/jqGD8S5xSmkTUa82E42Dcaq3EnA/lEcIQJHRZ4bAP+s8CgzpBFzckKl3hcQWCVFbeKiYcVqWZk1SJshxDYk83zJSg9R8EMY9ZCwMOBdQgC5jxlmM/jRY7aHsMDFE2Q2CFVUy+aAHKMpsPrHA9rlkEO27o1GWyQWU87KDmB/Y4cVe2BQMtghgx8Q7E6BXELC1En6yalIwU/JS/4k+mSCJbkkG5NxWUvI0Rht7GdyEkAS/yEDgNqj1ncg+1QTX2g2Ms98qZkSOmzPUz4o59I78B6Vt/EdFkEf5QD6O8/fehWuA9FmG8u0GGiZ2jP++mKmL3sJqH1eYyzqQzoyPVpjDKxRH02uJYNh/hw5iScJxTeMNBA8n90WSAMpB90knkwCtZOqVREPZnfxjzgN4QE8Qr+6pJ64fycfZmzh6LkbZ7EyOoCaDrf1BorRa9Xu5D2UUWFKihRil2HJCr5LgCqZqTITK813wkC9VL3O6RbPl8SGOBa/7neqeNeqf80HzZs3N9b9FQkFppskfOb+wBzWerZ9K86y/T/gyVgNRUcUNonsQHbz1as5x7ltEnfMCmkpfZktwmwzmiGUFeHoP23NCMwYp5Yd9X41cPusF7wQ7mInr6vqBVsfbNrAnGGZ5W6zPIiTRnTH1IrESHHYeMh7Dqr3x4ZWPHVQz0VNJ4OawiXEy3+hjxJX2DMk4w45Pf4YKP1pxT42ok0c+/RT2Hhof2vG4iUPPHReEOAxrWRM5Pf58+P+Qkil0LWaz7FK16tXm0/T6Vk0kqHDeEToLFPsb4m83xcC7vY/BsyB9BbCdIVzkO1Nfj5TFjIiW6h6jmReXz7PjsxkBuiz1mAsqrQZkk/l9sZqMZYbyrfBdEadEkKbt15+BCuthHS8wdJjTXk5kdoyLrEhnmjluF/amM6xm/1noDgNN72s6WxumT67jQ0am4LAFMvCSH2+pE7bzjAQ4408UKsQekQ7ogpoYD6sqeTm3kQ+Vr1OQXWaotNvFuQHn4DScX/MsKhqTvNC4qnCFBUYJE6DApZOrMgOxcxaveO/DjpkSoql6kvs/ePNsjc9V+psgYNx+kx/k3kVU0s1ttJgigCxn09X8b18ECGMwGHCznsLMMXP+guyaejp9Fe6iVDJRf44r54PZ88zDYu9i2ehdCnBby42KZiAC/2yI4Ywa+jPpuiyY62k2RBwCa1+MFfb4d03fMG87bBCnQ9QjIK+0GNvByxX43W2biXpvJRDEeXG9L5wxdMFDmRxZntRmzHBBzijC4zOxJ+tXLMSo+okBxjAnyPl83An6MuwJYVKgO8jMJIR5wxYm4DE6ZHzurYEE8RFc/a8n2nPMBk6mFNFAjckD6nVBjUJ+DpIALFzJ/lSi/8N0ByKyYlYuAIvumPrSTO4wtgrDBnLs4CejIoeYROacbKg3wQb2u3YzRx+wwq3cTLZR7Jk6oYszHyPHny5R7GVH6TtajAY2gU7mq/cTO7WZq8vK9QI8HJ+ArcntFFdOcfcuUozm+lns82I8vAtEN9jkIdTJbNPX7WbePbqMQwKiB3oJDz+3ZNUQRDkXdfAEBpOAVNz5B8P7YksHBwoqoprehEKEnldBqFijxV3/RDQiBZRWPpHs5BF6sPTypa4a4mthh5I8sbZ3Z1yACtdR1rEHUAEoORm/DRQ1x0EztimEXzhv3Us224w+Blv1n/fuEqB037hniWdPiL54gugyhYAKOaf0NKoUpPRrIiWH77rFcgVrljP5FltXLp+8g6ht0GRwBHw6UF2S83RmiPX0hqL1IHPdAY42JhlXco5FGAjBZRjQqKNKrc6Mumord8q1STurpQSq/tYX3y5wSCenXMhP0PzBKgGWsDCowo/Aly4LnfcHvvlCV6E/AKNNS0S7gsbqgltsSWHdKKeGb+YfS3v8USIl1H9C2+YQAC7UALgwdzFs/09vqTKozQFlntvV7TjjPPXxQJawQ1q6KvJbvbTh0yY+/TvSH6QjyhbO09t/rQcOBWGGShcibBBho+cyW9Br8MYMkCK9cJnASnAIHPqZE6LZfBm42VT9cTSRHXcm8mVGNpbPiLOiqOTKvoFUEsDBBQAAAAIAFuIVF2p8PDUIAIAACQEAAAUAAAAY29yZS9ldmFsL21ldHJpY3MucHltU8tu2zAQvPMrtrpEAlS5aW8OXCAIfCiK9NAYvRiGQFOrmDZFqiQVRzD8712RsqMi0UXi7uxjZsQkSVbcHaBBb6VwORywxwq2PSxfuOq4N/apRVHEfAFLLnbghLHowGgErt0RLdTWNPAFvIHbgrHlKxceGu4JK8wLWgdSe9QeFN+icnfgvO2E7yyNGicDjZMV9wh+h3D/dXb/jebssOGuYEmSMBZmlGXdDXVlCbJpjfW0gTaee2m0GzHCKIUiRAq+FRfgA1c0XuEVZbFAmlpMlhmhdadDfSmopsSBTg57R4Faoqrcf6GwOGOPgQcsrnPWa2qcD1Q3OdTKcL9hjFVYQ6gugz5pFHAOAYuvpLXHKhwz+Pw91s0Z0EMi3IKsgz6j7Pi348qFiMUaLWpBudpTihrItpX6GY476dG1fEh52Bq/A9QVqRq6rqjW8QZJj6blVhIj4A72ZutmgzqlMiTCbLJy0fbEylz+AsGt7WFwubisGd4WSVIdCYwsi7BUmsFicaV6iWWkzeNy9fvHw9McKil8VC+quiFZT7H7ZJFkPlUyj/l3LhHqvXMR+4HNhP7I/EnvYPelaTjk7Dwa+4y0TNg41aTpm42RRvTR2z5+TFQama+Hqk1I4qvA1sNP7JfWGjup4NIh/KHLiSGTXjPDUyedPmhz1OO9gtPQ8pM930EIz+F0k8NNsTdSp47+dazScXiWnZNrryxe6V90x9k/UEsDBBQAAAAIAFuIVF1MHvooxAIAALYGAAAUAAAAY29yZS9ldmFsL3JlY29yZHMucHmFlT1v2zAQhnf9ioO6yIHqJgXawYCDdPDaAi3QDkFAMNIpJkKRAnmKbaA/vkdSX3bSVoMsiff53kM6z/Nfe0kgDeCL1L0kZQ08I3Z+A9YgOKysq6FDB3iUbaeRbWugPYLnFQT7wktSa7BN+Nqu8zzPssbZFoRoeuodCgGq7awLaYylmMMPNrUkWWnpPfrJyNeqonJeKqFRqOvkQKdOmafR9os5ZVl2N5lm8Q67VOp39L2mTQZ8aWVwA8pQfMNjhxVhvQFPLn6Rxh/Qze+xuw002rI88A5Scm6bpH+GFsmpqoRrIAs30aFjg44E2Wc0fs6U4i4/h2hPaNBJLgDSAhwU7W1PMUOjjNSw+/YjFUK267AW1gi0HOHRWh1CBMvW1qgBTc2BFPkhWXhE3UTvvSKhVatodhxyh0EPsYFbbOVRGDwMhZZTQVMZtncVRn3gN3wNbGzjTxaX77h9hoRO8a3GBlg/hoeKUMoK3t/G/GkW4XLIaHAFvLpOJN1u4WZ9/eY4Gc3lLJP686wCuctZtcg8L/AMUg30+hQg6JZ6GZR8lH6QMzXAFZytN4rJX8aBSrYYqUgRz+RbjL+WHQ1cnesWIsffxrqLEng/BWETqCndhgH2dH8G9gNHijujYL0lfxCNrMi60zbYrrJhFCSVZv+wqe65ijLp9HfnYLiKvq43pFpc+vKGe7iY/6iX8Hv58dPntxBZKCGCkn6ynSPz7X81JdIiE4zA3tYTbGEOYtSqmCCrNJO8oKX8t54lXF2xNU8+nCshQAT3Er9wqQb4KJvDTQtRN6l4lD8DlTvnrCtyY2dw+MSIbOarySkCzM37vi1w2A4BC2SOJscVfACNppjeL/cSd1ukZrfj+RQDb+N97n07Poz9Tqg0cHDWPM179g2RXm3h+0WtcTtPnY4irYez4GHOQ1aEmc6ZzhF7lST9KSTz7A9QSwMEFAAAAAgAW4hUXa/j4OgGBwAAkRQAABcAAABjb3JlL2V2YWwvc3RydWN0dXJlZC5wecVYW2/bNhR+96/ghD3IgOom6bCixgw0vQ3dumRokr4YnkBLtK1aojSSauJl/u8755CSaMkJim7A/GKJ5Ll/50IFQXBlVJYYlpTSKJ4YzbhMWSFwVbNVqZjZCPZFKJ2VUqRM7yQsmCxh52dPz58xw/VWT4IgGI1WqixYHK9qUysRxywrqlIZ4CdLww2Q69HIrX3WpbTnU26AWyGa0/hud8yuyuS6Wf+QGaF47qRUu5RLVMLtvuJa/FamIo/Y61KusvUbsCli7zKRpxGzJr4qS9he4VL8hecZSCrVaDRKcq61O/NRJKVKw5bdeDpi8CvwOU6IM5t5IkJxB16bBeCnZZYGEdPEZnatajFueV9nyVaY0BfhGBvaibN0ipTAmjQOK27AWjlTwR/XT+YnT14s7n/cfx+MiaZSZVonZtq4ZB6osoYnkB5UKpPuUSfgeHhcOKKsVJnZeVR5eYvnZKkKnuPTJltv3PGE5/mSJ9up5zvaSGsRY4isvn+zCwDFiHZe9jwbBs1Zp/dLcgYga1OmlpdYMTpOp8Ik1xG+g+esc/CXrewSyzQDGJE8QiiSTBALmS5XaIIJLe2kWxiz72aWvGOIP8UzLdgn3HirFKp6o5Ere391iaYLmXLFPNWJSACspWXXRRZSZnupUqGORbfEjceCe3kQ3Ibpay4TkVuunQDHUwkOueOFMa2rPEtQVwhhsuFyLdK4yCSiMbhVpVzHcLSAwLZQTzYirXPxO4CvroYi2vjSW14mlLyeSHC7NHEq9BZl5CVPIVHjJd/ZV4A0wu4hVPx7RPzfsX93c/H6+v3lxRXE9J4OBVg7tzEFPJgyz6fgkZ+FoSIKNSSDQroD13JTa1auUC7RTIJxZBklFPyOkwcGYGXfWrK2PutKJBm4OnX46PhpF+y4omgjy8P4A1fI7m3H0x5kHCo3+YHSrUEBMd63UHpXywSXX0O9OJYBkhfCA47vpahnazTU1dYirtZ1IaTRU5YCf5CNMLGFNsZGEhqowg12YavimdIhgl57uFFC17nBiO3bNXTfVuyipsZIRlSHiAHAwRnctCwOd49j6k2TlOyXq8sLpPfQ1Gkzh40Fczjto80eGbVmZdKmBrQhAJBsMO9Z2FdjFby3NFaLhnDK7ol0jzXHE4i+nGA2a/JoxMrlZwE+Jn/GGwDJjB4j8LDSolVk1tds7GLksl7EXOpbKGY4K0TMi5Z1+2wYzJFzPFKw2YwFuBVTsyW8BdO+t2yPndhO3QhuyoJdTeuiCoesVw7DMTY9KGQeb1zBuu2h/LiElgLAqpGiqRBzZDHBNFjMTxZ9Ytps8X1ET8/A+wC5QP62HCFjWlpYR9EW2keAcCO3sryV6GmYHWA+S8kBAAX8IyRQyMjPpF9ogxYxcQfFxYh2ZFG7zkF+4Oz5gd6nkxNaEneJqAwLO7Uidr2rmkcoGzXNmPQ+HsT3BNi0me9MAEAAHBysHlBXiZVQAuqMTbMjeGwJhuY52StICkq3I9TOakRSK2r83xpMMaE+qp3FD9pKlAMHDbLniLecrINk+EZpg4R6WJxHrOui4Gp34FY9EAqJi2Og24aOxaBjOw019Xx87xEdr9EfxZ91pqC95dkabjYRMJaiqMwOhmWBnQZvLNT82sBqV8VVeYtZPrcNCrtIY2AHN+gXf2VV+JAhFmu12TyIyx6cnNx25Gh+ga29GGIoAkfyt+MTHRJiuy24PQ2kALjeAYpnDNNUsunv7z0jdocdsSnqj+bLYTMsbyd1RRXR12kGpQP91ioxaxMRBcys/8aHvL6uaXiSYZylEZUnkI882QXYkAGMVsx8u2gl4TNNDBhbK5s9JbwN+LbRHewcaityLR7QCiv8oVKe9XY78HRzCx37by0/+KtgtvNBpye8qmAyDuHZSoDBRR9A0YdhB5NDjHnrPrS85V4oHo0mOY/1/GR57f0C1ekISk8ptAi4NdzIafYDXI8prphdOOWBvU1g7bMbE3FTlwpyGAn12Ikaja7Pr36N3346/3Bzfn350bsVLLncQgF8/hzH7gCv5nBvyvlS0H37wAfs2Vkzsg9gC7TuQ4FN71OkHnQFWDx90TIZTjXAxV5IDVzmteNypN63fPb9MU4g9ugWZ3MZbx0OOU11A9t73iBH4/l28mrPwp2e7vPg3hB5uamGHu0XKKrozfn59GzxWEG/kbquKgoQoeZpq2/7dStolcAuQoLwqwfe0nW8FHgxR200qEybMOiCSoZ6yxMYwQp+F0txCy7cCqmD/deq02nScXS65FmR4bWkJ4/c1pfXdZH52aKbZCGrQ2Izbj6TANTQDnw8ZT/NnBB4OD05++ExpQ8FsqLWhi3pfozoXeOVE7+/nTJTEq+mIbpZgCLnPpZVu9AW9dl94Fk9HbpxavXb78ejfwBQSwMEFAAAAAgAW4hUXbUjMSxUAAAAZAAAABAAAABkZXBlbmRlbmNpZXMudHh0JcsxDoAgEAXRnsP8uIsaLfYwxCwRjGCAytOL2r7MxJCiYxELwmwu9U1kABPYVOe1aaq51NcWDKblQ1O49Re2sJ3KtoswaASZVlyqPpfzSybQ2q8HUEsDBBQAAAAIAFuIVF1D/bukmQYAAB4MAAAPAAAAZXhwZXJpbWVudC5qc29ulVZLc9vIEb77V6B42YtFzgvzWJ822RxSqSTeOHtW9cz0kLBIAAFAWcqW/3u+ASlRZfuSUokkHtOPr7u/r/941zSbLm9+bjYPtN8f+Y4f6XimpRv6u0d1p4SyUiixeV9fjMOwzMtE4z2duM/4X+rJfx+4GSfOnHieh6kp1B05N13fcD+fJx67sYlchombzCMOcp+e8Xhe6HhcXTU4dRoyH5vjQLnr99vm95mbBYb7YWH4ffhpbqqdL91yaO7uxuflgGPL8GIG37gguDyNC4zPAywjiEfuHz809YXV/vsm00Lvmz33PF1cU5+bkRLSRxaw1hDiPPfpQP2e8/aS+XTul+7EyPYPXOJGQYrz/Xwg1drXu7ifhr50++3neegrNNaKHHV0OjvvpQxJmFa3bHwqIXJb2jYmLYXwRpKT3mpHbEMbjVFFuZxX76vhW8T33/hQWrWZRJFtjN54FtKnllg4GV0wWmvFoXgns41WsOKsVKLgskkxBJ3VzceJpz3P2+VpLav3WrIpkoQxxkltRHFJkEcuMeNaWGqjt45DW7J3eGqySK0WnoIwuX1jt0K/nanwgo4YprmaL8YVJ6VzRauYshQIt3WaU5BOMmy7YorJWkrjs/asvYLLYlQI1jsTb+aX4YH77r88vUJCHKXWwgFTStkX9sBW5mAot4ai9SEX57SW3ovAnHWKDKwAe1CZ2PzA9reo5zaLUISLqCVStqkVPmqRJeKXJmYpfZCmjc5w8IqURd1FCkrXgJDszcXjkCi+2k0kRQamJWrO0reuzVmycg6d5kKSFkgBjNoeRgMJQ4UESxGlMSJIsVnNfr1YvyBfzf72hftd/dB3Ymv/dPW+uTb+/LaFP3f9Z1L1kN7Krb0FOnJZG0Nsldy+aZtvCiu2fit+gOD1odJb/fbplA5rF2+l2b5BBTTTz+CM0/Vgu5Vh+012Ez92c3fFDXiHzMoH9GuWCkOFyXMCYBvfJtQbY+cSvWQ+9zTOh2FNaAeymebdgaYj9Sd62m0TpQPvDuf9HmRUKNXfcbeiOd/dVRwvnxc0dy/G5t3/G8YCvqpMdaOSDbUts4wS3CGVj04E75RmQZiSgNElAS4RoOVWJqsyPiN7l0MqMmAE/YvpR+R0AUe+u2J26Yd75JYexqG70PfHru/BlJHAuFeS7EGlmcaFpwv93cjnxoD7CYPzjOtlOvNLNoeufwBiuFvoOL/cBqT3PX+5X1uhVlOr14jmNIy8hjEN88hp6R65+duqRU0aTiNN4HKw9J4q0zf81M0LPDR///ipseaOnwgAXvSnSzXC+QNU5pGPw1j16cr3SAgN1fX1ZFWorqdjs/C8XPmdllqIpcYmL2EtwzheMtn8i9Mw5QZGp+cmdzPV1KvxbfMP2D1MjNofc3ORjLk6oPMynABYaiZecIoKsHyrOhCtaZmv3m95Vnd/eaK0vEloVagTzMBa7krhCfoJN7W+53kFor5Bzcywgl46PsNrDRlV/fPvv/4CHA/DtHyoKKQjdadmKDc4IigAmtzwf84dtL/avobFTyOO1ZB+Hb70VZmbB556qPRwXsbzMtfch/P+8FKvXz7+9X0V7b45j+vrR1DbEeFAml8L98/4GVVuPmHyq+SesQ5MzWX52F13kNqKdzewXreQ3Qr4J40wnue6X1z8XsP9oSRvvgwTgt6OzyuhMclA2gaWOjmwpgzZ+laoILTGzONJkpygx55TaqV2RbV1XI1MySZ+Ga11gFbKXsk1QkGUw6RyCU4WDdHUklhFkjomPJCaNUbV6pwgcC4IaIwUbLMP4PQXKt28bkfdTYUzxB36EANrMIg1hYsmJSkaWwQ0XMnsRNIBBIMXk2cfyYIQQsEKEFPavM5g119R8MGKtkDvsmutg/5FoaVLkPvWCqdIKxW9LK1NWQAbpYstJRSk42Ly6oYCNrpd3Rd3l+6cX1DW2SilikkmCdsWiqQBBCN5oVtkQIqggkYLsKWwzqVCWIcsw3tWNnxv/9oLfHWQBLQxKAWBN0hSC+EKWBabFJiWZOEkc0sSfyI4XYCGQBwev4JBWvp7B5d5eU3AYV1F2X0yQhmvtbMt2kKVKIiyxprFhE1OpITCisJSqkK6TUWzN4bT9/axNZ/Tgl04X13AgEWryISAtMO+hnx8cSB3ZSAT6ED2HjW3ASAV6UhG07IMLRYZysp/76IS3dV4a6UHwDGjF5K1WEKw+4SIRSU6LIMQEFbYirB7CcnOJ7IOTySKAikhclVlv777+u5/UEsDBBQAAAAIAFuIVF2ZcvJdUQEAAEkCAAAHAAAAbWFpbi5weU1RTW+DMAy951dEuQwkPrrduUzapN162K2aUAjusApJlBhK//0MlLUcwrNf7PdiK6W+bCTd95I6kB6thVaG0RIOIN1IEVtYKeMsabQQJA76F16iPN6oc1Z6bS6ciIVSSggcvAskXdxRHBsfnIH4yNz+4QR2EufgBu5CXY+NvBNHDoUIzpGs1iCp6zP2UNdp4XUAS4JLMTg7MN7vqJIGX1qYKYfZQ8CFzO+vUalY5IoPO72P2LcQkitSV3v01afuI2SLM4KhjkhQ78/auLQwATRB8qSaCr9NoJKRwjMjS6katOXGs/BjCAW7SYTk78RqBcxgRtJNz+oqHxSf7Gf55fm9OpMb4Bxuq9po63KjTQd5i2HNLOdiZJ0aO2jBg23BGuTl0Ewq/clWZS4yl+o7jLDFy3R41dXr2+GQiVS4zdeU7MKnHTy3v7pwgVD4G/dNxR9QSwMEFAAAAAgAW4hUXVa0DlgNBAAArwkAAAkAAAB3b3JrZXIucHmdVV1v2zYUfdevIPgkoQYbONgeXPjBjd3NQNAEqd09uAJBS5TFRSI1koqzFf3vuyQl2Y7dtJteDJPnfp97KOpGaYuUiZQhXD4JreQGf/y8nC9ndPXhekzvPi8eHpbzBU7RFOErfAJc3T3c/E5nt7d3fwT0zfr97ezTuZEIcUpmykps+7/hBw5IzS3LmWX9zZ9GyajQqkbuNKuYMdx0eMRMLjIbrhtmncf+6h7+9j6s0lkZUGW72wm5K1jGadkOaCNZY0plaa72slIsD+hMaU74E6uIC96DNWc55c+sbipuXgJ3XHLNLB9SlGbP9Qg5p7RWOa9eWkDFWmRDTeA4s7RmFlKOtFIWGueKiSktRMUpTUjDNJc2aiom4dJ1iDj3Jo49/i3C/LnhWtSAIu4aJ8QnbfmzjZMkKpRGktV8hHKx4wb6L5HztsEuhKGmZONffsUpEZbXJk4mEYLPtf4wOhJAQ0znrwuz/dtysEpIyZ9DgDhB02kXLOr8+KmQrM0ZEYayJyYqtq14nIwQnqGb9XyGfrtfI2Gg4X+1QvP8HZIK3cBZwapqy7JHHAUnhltaQAssEA86V7cVbTTPhBFKxrgUuxLi4qRDO0sucxNiBzwBj2pPbXE9hpZ+YJXhF9BS/hyOwWK4WiApaBY1eRN76OUMOmzNa8qLQmQCxvazNkAMD13pFpANYBj0GFL76uYxubBY5Ilr3xc/MNRT4UAB3UoL1MHpBvf+cPrtiDNA2jbgu+uLNPEoYppK2Bi/wcnmKnUceCXGxrlPo34Xe96f7WZ85sPvFU5HwJQnP/TpGaS/wemr/D+y+B+rMKT+X9bBJz+CdYDxin+cVlAo/aAX8UclIdMtMzwcTI3VQyioxh8Sq2LsqAG77mQlTkA89o4Jm9TX2wmWK/VEwAbNyEqePXq5qHBX5l7YsttTIQsOqpOFHPo+uA/CQJQgxXGQu/i8pi4c7LZPcnSscyN0PYY6nDNROH8b2FlLK1ELi9OTSBtsnHL65+SKXEXdsSGsaWA7oJh9EqnWNu3An5P3bPF+uYZXab26X6/ofPkAdOjgpH7MhY6DtpqpWyiXpDCWqsdp2K+48wzdAn1xBQOjTC+xew0cCRrrNTlv68a1d29G0PUc3E7HCXqD8BdAR5qbtnJJfsX9KPAEVVx6CyeCRx2CG9PWsT6U70aq/TAd+u2JnfK0BdjByiroT06VpHD7urlv+0sHR/N4zXbXtGBzpOw7UOYcVg944zbCi7snQI/q1Mij4W5Qg8mgLnDKrIXuue2doMD2TMlC7AiFG0mF65577JjDjPxwgvgDHi8XiwXqXod3yCkm+jS/n2GPU1Zlquq3fPK9x+17D+ql1f52TJMw5R8wxGMucaTRQl6Awq78C1BLAQIUAxQAAAAIAFuIVF2HAVEGKQsAAL6SAAALAAAAAAAAAAAAAACAAQAAAABjaGVjay5qc29ubFBLAQIUAxQAAAAIAFuIVF0SzYRzoAMAAJwJAAARAAAAAAAAAAAAAACAAVILAABjb3JlL2V2YWwvZGF0YS5weVBLAQIUAxQAAAAIAFuIVF3KEpH+QgkAALYZAAAVAAAAAAAAAAAAAACAASEPAABjb3JlL2V2YWwvZ2VuZXJhdGUucHlQSwECFAMUAAAACABbiFRdqfDw1CACAAAkBAAAFAAAAAAAAAAAAAAAgAGWGAAAY29yZS9ldmFsL21ldHJpY3MucHlQSwECFAMUAAAACABbiFRdTB76KMQCAAC2BgAAFAAAAAAAAAAAAAAAgAHoGgAAY29yZS9ldmFsL3JlY29yZHMucHlQSwECFAMUAAAACABbiFRdr+Pg6AYHAACRFAAAFwAAAAAAAAAAAAAAgAHeHQAAY29yZS9ldmFsL3N0cnVjdHVyZWQucHlQSwECFAMUAAAACABbiFRdtSMxLFQAAABkAAAAEAAAAAAAAAAAAAAAgAEZJQAAZGVwZW5kZW5jaWVzLnR4dFBLAQIUAxQAAAAIAFuIVF1D/bukmQYAAB4MAAAPAAAAAAAAAAAAAACAAZslAABleHBlcmltZW50Lmpzb25QSwECFAMUAAAACABbiFRdmXLyXVEBAABJAgAABwAAAAAAAAAAAAAAgAFhLAAAbWFpbi5weVBLAQIUAxQAAAAIAFuIVF1WtA5YDQQAAK8JAAAJAAAAAAAAAAAAAACAAdctAAB3b3JrZXIucHlQSwUGAAAAAAoACgBrAgAACzIAAAAA"
)
assert (
    hashlib.sha256(raw).hexdigest()
    == "9667b6346faf09491b192312a9f1bee3821110272497a45d763ffbce6a0c9382"
)
root = pathlib.Path("/kaggle/working/frozen-eval")
root.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for item in archive.infolist():
        p = pathlib.Path(item.filename)
        assert not p.is_absolute() and ".." not in p.parts
    archive.extractall(root)
os.environ["NEBIUS_OUTPUT_DIR"] = "/kaggle/working/evaluation"
subprocess.run([sys.executable, str(root / "main.py")], check=True)